In [1]:
from collections import OrderedDict
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from datasets.utils.logging import disable_progress_bar
from torch.utils.data import DataLoader
import torchvision.models as tvm
from torchvision.models import MobileNet_V2_Weights, ViT_B_16_Weights
import flwr
from flwr.client import Client, ClientApp, NumPyClient
from flwr.common import Metrics, Context
from flwr.server import ServerApp, ServerConfig, ServerAppComponents
from flwr.server.strategy import FedAvg
from flwr.simulation import run_simulation
from flwr_datasets import FederatedDataset

DEVICE = torch.device("cpu")  # Try "cuda" to train on GPU
print(f"Training on {DEVICE}")
print(f"Flower {flwr.__version__} / PyTorch {torch.__version__}")

Training on cpu
Flower 1.31.0 / PyTorch 2.12.0+cpu


CIFAR-10 enables us to train image classifiers that distinguish between images from ten distinct classes:

- **‘Airplane’**
- **‘Automobile’**
- **‘Bird’**
- **‘Cat’**
- **‘Deer’**
- **‘Dog’**
- **‘Frog’**
- **‘Horse’**
- **‘Ship’**
- **‘Truck’**

In [2]:
NUM_CLIENTS = 10
BATCH_SIZE = 32
import torch
from torchvision import transforms

pytorch_transforms = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        ),
    ]
)

def apply_transforms(batch):
    batch["img"] = torch.stack([pytorch_transforms(img) for img in batch["img"]])
    batch["label"] = torch.tensor(batch["label"])
    return batch

def load_datasets(partition_id: int):
    fds = FederatedDataset(
        dataset="cifar10",
        partitioners={"train": NUM_CLIENTS},
    )

    partition = fds.load_partition(partition_id)

    partition_train_test = partition.train_test_split(
        test_size=0.2,
        seed=42,
    )

    partition_train_test = partition_train_test.with_transform(apply_transforms)

    trainloader = DataLoader(
        partition_train_test["train"],
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,   # 🔥 IMPORTANT FIX
    )

    valloader = DataLoader(
        partition_train_test["test"],
        batch_size=BATCH_SIZE,
        num_workers=0,
    )

    testset = fds.load_split("test").with_transform(apply_transforms)

    testloader = DataLoader(
        testset,
        batch_size=BATCH_SIZE,
        num_workers=0,
    )

    return trainloader, valloader, testloader

In [3]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from torchvision.models import vit_b_16, ViT_B_16_Weights

In [ ]:
# %%
"""cFedLoRA on CIFAR-10

Single-notebook starter for:
- CNN baseline
- MobileNetV2 + LoRA
- ViT-B/16 + LoRA
- FedAvg
- FedProx
- FedCluster
- FedDANE (approximate)
- FeSEM (multi-center EM-style)
- FedAvg+LoRA
- cFedLoRA

This notebook is self-contained and uses torchvision CIFAR-10 data with a custom Dirichlet partitioner.
The federated orchestration is implemented in pure PyTorch so the notebook is not tied to a specific Flower runtime version.
"""

# %%
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Callable, Iterable
from copy import deepcopy
from collections import OrderedDict
import math
import random
import os

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.datasets as tvdatasets
import torchvision.models as tvm
from torchvision.models import MobileNet_V2_Weights, ViT_B_16_Weights
from torch.utils.data import DataLoader, Subset, Dataset

# %%
# -----------------------------
# Global configuration
# -----------------------------

@dataclass
class Config:
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    data_root: str = "./data"
    num_clients: int = 10
    num_classes: int = 10
    alpha: float = 0.5
    batch_size: int = 32
    local_epochs: int = 5
    finetune_epochs: int = 1
    rounds: int = 20
    lr_cnn: float = 1e-3
    lr_mobilenet: float = 5e-3
    lr_vit: float = 5e-5
    weight_decay: float = 0.0
    lora_rank: int = 8
    num_clusters: int = 3
    proximal_mu: float = 0.1
    sample_clients_per_round: float = 1.0  # use all clients by default
    test_fraction_per_client: float = 0.2
    personal_fraction_per_client: float = 0.1
    pretrained: bool = True
    download_data: bool = True
    max_eval_batches: Optional[int] = None

CFG = Config()


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CFG.seed)
DEVICE = torch.device(CFG.device)
print(f"Training on {DEVICE}")

# %%
# -----------------------------
# Utility functions
# -----------------------------

def count_parameters(model: nn.Module, trainable_only: bool = False) -> int:
    return sum(p.numel() for p in model.parameters() if (p.requires_grad or not trainable_only))


def get_state_copy(model: nn.Module, only_trainable: bool = False) -> OrderedDict:
    if only_trainable:
        return OrderedDict((k, v.detach().clone()) for k, v in model.state_dict().items() if k in dict(model.named_parameters()) and dict(model.named_parameters())[k].requires_grad)
    return OrderedDict((k, v.detach().clone()) for k, v in model.state_dict().items())


def set_state(model: nn.Module, state: Dict[str, torch.Tensor], strict: bool = True) -> None:
    model.load_state_dict(state, strict=strict)


def clone_model(model: nn.Module) -> nn.Module:
    return deepcopy(model)


def state_dict_subset(state: Dict[str, torch.Tensor], keys: Iterable[str]) -> OrderedDict:
    keys = list(keys)
    return OrderedDict((k, state[k].detach().clone()) for k in keys)


def average_state_dicts(states: List[Dict[str, torch.Tensor]], weights: Optional[List[float]] = None) -> OrderedDict:
    if not states:
        raise ValueError("states is empty")
    keys = states[0].keys()
    for s in states[1:]:
        if s.keys() != keys:
            raise ValueError("All state dicts must have the same keys")
    if weights is None:
        weights = [1.0 / len(states)] * len(states)
    total = float(sum(weights))
    weights = [w / total for w in weights]
    out = OrderedDict()
    for k in keys:
        out[k] = sum(w * states[i][k].detach() for i, w in enumerate(weights))
    return out


def model_vector(model: nn.Module, only_trainable: bool = True, name_filter: Optional[Callable[[str], bool]] = None) -> np.ndarray:
    vecs = []
    for name, param in model.named_parameters():
        if only_trainable and not param.requires_grad:
            continue
        if name_filter is not None and not name_filter(name):
            continue
        vecs.append(param.detach().cpu().flatten().numpy())
    if not vecs:
        return np.zeros(1, dtype=np.float32)
    return np.concatenate(vecs).astype(np.float32)


def cosine_similarity(a: np.ndarray, b: np.ndarray, eps: float = 1e-12) -> float:
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na < eps or nb < eps:
        return 0.0
    return float(np.dot(a, b) / (na * nb))


def topk_clients_by_size(client_sizes: Dict[int, int], frac: float) -> List[int]:
    client_ids = sorted(client_sizes.keys())
    if frac >= 1.0:
        return client_ids
    k = max(1, int(round(frac * len(client_ids))))
    return random.sample(client_ids, k)

# %%
# -----------------------------
# CIFAR-10 transforms and partitioning
# -----------------------------

def get_transforms(model_name: str) -> Tuple[transforms.Compose, transforms.Compose]:
    model_name = model_name.lower()
    if model_name == "cnn":
        train_tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616)),
        ])
        test_tf = train_tf
        return train_tf, test_tf

    # Pretrained torchvision backbones: 224x224 RGB, ImageNet normalization
    train_tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])
    test_tf = train_tf
    return train_tf, test_tf


def load_cifar10(download: bool = True) -> Tuple[Dataset, Dataset]:
    trainset = tvdatasets.CIFAR10(root=CFG.data_root, train=True, download=download)
    testset = tvdatasets.CIFAR10(root=CFG.data_root, train=False, download=download)
    return trainset, testset


def dirichlet_partition_indices(labels: np.ndarray, num_clients: int, alpha: float, seed: int) -> Dict[int, List[int]]:
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels)
    classes = np.unique(labels)
    client_indices: Dict[int, List[int]] = {i: [] for i in range(num_clients)}

    for c in classes:
        class_idx = np.where(labels == c)[0]
        rng.shuffle(class_idx)
        proportions = rng.dirichlet(np.repeat(alpha, num_clients))
        proportions = np.array([p * (len(client_indices[i]) < len(labels) / num_clients * 1.2) for i, p in enumerate(proportions)])
        if proportions.sum() == 0:
            proportions = rng.dirichlet(np.repeat(alpha, num_clients))
        proportions = proportions / proportions.sum()
        split_points = (np.cumsum(proportions) * len(class_idx)).astype(int)[:-1]
        splits = np.split(class_idx, split_points)
        for i, part in enumerate(splits):
            client_indices[i].extend(part.tolist())

    # Ensure every client has some data; move a few samples from largest clients if necessary.
    empty_clients = [cid for cid, idxs in client_indices.items() if len(idxs) == 0]
    if empty_clients:
        donors = sorted(client_indices.keys(), key=lambda cid: len(client_indices[cid]), reverse=True)
        for cid in empty_clients:
            for donor in donors:
                if len(client_indices[donor]) > 1:
                    client_indices[cid].append(client_indices[donor].pop())
                    break

    for cid in client_indices:
        rng.shuffle(client_indices[cid])
    return client_indices


class IndexedDataset(Dataset):
    def __init__(self, base: Dataset, indices: List[int], transform: Optional[Callable] = None):
        self.base = base
        self.indices = list(indices)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int):
        x, y = self.base[self.indices[idx]]
        if self.transform is not None:
            x = self.transform(x)
        return x, y


def split_client_indices(indices: List[int], personal_fraction: float, seed: int) -> Tuple[List[int], List[int]]:
    rng = np.random.default_rng(seed)
    idxs = np.array(indices)
    rng.shuffle(idxs)
    n_personal = max(1, int(round(len(idxs) * personal_fraction))) if len(idxs) > 1 else 0
    personal = idxs[:n_personal].tolist()
    train = idxs[n_personal:].tolist()
    return train, personal


@dataclass
class ClientData:
    trainloader: DataLoader
    valloader: DataLoader
    personalloader: DataLoader
    n_train: int
    n_val: int
    n_personal: int
    label_hist: np.ndarray


def build_federated_cifar10(model_name: str, num_clients: int, alpha: float, batch_size: int,
                            test_fraction_per_client: float = 0.2, personal_fraction_per_client: float = 0.1,
                            seed: int = 42, download: bool = True) -> Tuple[List[ClientData], DataLoader, np.ndarray]:
    trainset_raw, testset_raw = load_cifar10(download=download)
    train_tf, test_tf = get_transforms(model_name)

    train_labels = np.array(trainset_raw.targets)
    client_map = dirichlet_partition_indices(train_labels, num_clients=num_clients, alpha=alpha, seed=seed)

    client_data: List[ClientData] = []
    for cid in range(num_clients):
        train_idx, personal_idx = split_client_indices(client_map[cid], personal_fraction=personal_fraction_per_client, seed=seed + cid)
        n_val = max(1, int(round(len(train_idx) * test_fraction_per_client))) if len(train_idx) > 1 else 0
        rng = np.random.default_rng(seed + 10_000 + cid)
        rng.shuffle(train_idx)
        val_idx = train_idx[:n_val]
        main_train_idx = train_idx[n_val:]

        train_ds = IndexedDataset(trainset_raw, main_train_idx, transform=train_tf)
        val_ds = IndexedDataset(trainset_raw, val_idx, transform=test_tf)
        personal_ds = IndexedDataset(trainset_raw, personal_idx, transform=test_tf)

        labels = train_labels[np.array(client_map[cid])]
        hist = np.bincount(labels, minlength=CFG.num_classes)

        client_data.append(
            ClientData(
                trainloader=DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0),
                valloader=DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0),
                personalloader=DataLoader(personal_ds, batch_size=batch_size, shuffle=True, num_workers=0),
                n_train=len(train_ds),
                n_val=len(val_ds),
                n_personal=len(personal_ds),
                label_hist=hist,
            )
        )

    testset = IndexedDataset(testset_raw, list(range(len(testset_raw))), transform=test_tf)
    testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=0)
    return client_data, testloader, train_labels

# %%
# -----------------------------
# Models
# -----------------------------

class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def freeze_module(module: nn.Module) -> None:
    for p in module.parameters():
        p.requires_grad = False


class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int = 8, alpha: float = 1.0, dropout: float = 0.0):
        super().__init__()
        self.base = base
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.lora_A = nn.Linear(base.in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, base.out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)
        freeze_module(self.base)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.base(x) + self.lora_B(self.lora_A(self.dropout(x))) * self.scaling


class LoRAConv2d(nn.Module):
    """Channel-bottleneck adapter for arbitrary Conv2d.

    This is a practical approximation of conv LoRA suitable for torchvision backbones.
    The base convolution is frozen; the adapter learns a residual path with the same
    spatial stride/padding as the base convolution.
    """

    def __init__(self, base: nn.Conv2d, rank: int = 8, alpha: float = 1.0, dropout: float = 0.0):
        super().__init__()
        self.base = base
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()
        k = base.kernel_size
        s = base.stride
        p = base.padding
        d = base.dilation
        self.lora_A = nn.Conv2d(
            in_channels=base.in_channels,
            out_channels=rank,
            kernel_size=k,
            stride=s,
            padding=p,
            dilation=d,
            groups=1,
            bias=False,
        )
        self.lora_B = nn.Conv2d(
            in_channels=rank,
            out_channels=base.out_channels,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=False,
        )
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)
        freeze_module(self.base)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.base(x) + self.lora_B(self.lora_A(self.dropout(x))) * self.scaling


def replace_module(parent: nn.Module, child_name: str, new_module: nn.Module) -> None:
    setattr(parent, child_name, new_module)


def inject_lora(module: nn.Module, rank: int, target_types: Tuple[type, ...], name_filter: Optional[Callable[[str], bool]] = None,
                exclude_names: Tuple[str, ...] = ()) -> None:
    for name, child in list(module.named_children()):
        full_name = name
        if name_filter is not None and not name_filter(full_name):
            inject_lora(child, rank, target_types, name_filter=lambda n, p=full_name: name_filter(f"{p}.{n}"), exclude_names=exclude_names)
            continue
        if full_name in exclude_names:
            continue
        if isinstance(child, nn.Linear) and nn.Linear in target_types:
            replace_module(module, name, LoRALinear(child, rank=rank))
        elif isinstance(child, nn.Conv2d) and nn.Conv2d in target_types:
            replace_module(module, name, LoRAConv2d(child, rank=rank))
        else:
            inject_lora(child, rank, target_types, name_filter=(None if name_filter is None else lambda n, p=full_name: name_filter(f"{p}.{n}")), exclude_names=exclude_names)


class DualLoRALinear(nn.Module):
    """
    Frozen base linear layer + two independent LoRA branches:
      - local branch   : trainable on each client
      - cluster branch : set by server, frozen on client
    """

    def __init__(self, base: nn.Linear, rank: int = 8, alpha: float = 1.0, dropout: float = 0.0):
        super().__init__()
        self.base = base
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        # Local adapter
        self.lora_local_A = nn.Linear(base.in_features, rank, bias=False)
        self.lora_local_B = nn.Linear(rank, base.out_features, bias=False)

        # Cluster adapter
        self.lora_cluster_A = nn.Linear(base.in_features, rank, bias=False)
        self.lora_cluster_B = nn.Linear(rank, base.out_features, bias=False)

        nn.init.kaiming_uniform_(self.lora_local_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_local_B.weight)

        nn.init.kaiming_uniform_(self.lora_cluster_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_cluster_B.weight)

        freeze_module(self.base)

        for p in self.lora_cluster_A.parameters():
            p.requires_grad = False
        for p in self.lora_cluster_B.parameters():
            p.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_d = self.dropout(x)
        local = self.lora_local_B(self.lora_local_A(x_d)) * self.scaling
        cluster = self.lora_cluster_B(self.lora_cluster_A(x_d)) * self.scaling
        return self.base(x) + local + cluster


class DualLoRAConv2d(nn.Module):
    """
    Frozen base conv layer + two independent LoRA branches:
      - local branch   : trainable on each client
      - cluster branch : set by server, frozen on client
    """

    def __init__(self, base: nn.Conv2d, rank: int = 8, alpha: float = 1.0, dropout: float = 0.0):
        super().__init__()
        self.base = base
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

        k = base.kernel_size
        s = base.stride
        p = base.padding
        d = base.dilation

        # Local adapter
        self.lora_local_A = nn.Conv2d(
            in_channels=base.in_channels,
            out_channels=rank,
            kernel_size=k,
            stride=s,
            padding=p,
            dilation=d,
            groups=1,
            bias=False,
        )
        self.lora_local_B = nn.Conv2d(
            in_channels=rank,
            out_channels=base.out_channels,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=False,
        )

        # Cluster adapter
        self.lora_cluster_A = nn.Conv2d(
            in_channels=base.in_channels,
            out_channels=rank,
            kernel_size=k,
            stride=s,
            padding=p,
            dilation=d,
            groups=1,
            bias=False,
        )
        self.lora_cluster_B = nn.Conv2d(
            in_channels=rank,
            out_channels=base.out_channels,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=False,
        )

        nn.init.kaiming_uniform_(self.lora_local_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_local_B.weight)

        nn.init.kaiming_uniform_(self.lora_cluster_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_cluster_B.weight)

        freeze_module(self.base)

        for p in self.lora_cluster_A.parameters():
            p.requires_grad = False
        for p in self.lora_cluster_B.parameters():
            p.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_d = self.dropout(x)
        local = self.lora_local_B(self.lora_local_A(x_d)) * self.scaling
        cluster = self.lora_cluster_B(self.lora_cluster_A(x_d)) * self.scaling
        return self.base(x) + local + cluster


def inject_dual_lora(
    module: nn.Module,
    rank: int,
    target_types: Tuple[type, ...],
    name_filter: Optional[Callable[[str], bool]] = None,
    exclude_names: Tuple[str, ...] = (),
    prefix: str = "",
) -> None:
    for child_name, child in list(module.named_children()):
        full_name = f"{prefix}.{child_name}" if prefix else child_name

        if child_name in exclude_names or full_name in exclude_names:
            continue

        if isinstance(child, nn.Linear) and nn.Linear in target_types and (name_filter is None or name_filter(full_name)):
            setattr(module, child_name, DualLoRALinear(child, rank=rank))
        elif isinstance(child, nn.Conv2d) and nn.Conv2d in target_types and (name_filter is None or name_filter(full_name)):
            setattr(module, child_name, DualLoRAConv2d(child, rank=rank))
        else:
            inject_dual_lora(
                child,
                rank=rank,
                target_types=target_types,
                name_filter=name_filter,
                exclude_names=exclude_names,
                prefix=full_name,
            )


class MobileNetV2LoRA(nn.Module):
    def __init__(self, num_classes: int = 10, rank: int = 8, pretrained: bool = True):
        super().__init__()
        weights = MobileNet_V2_Weights.DEFAULT if pretrained else None
        self.model = tvm.mobilenet_v2(weights=weights)

        freeze_module(self.model)

        # MobileNetV2: adapt all Conv2d layers, keep classifier trainable.
        inject_dual_lora(self.model, rank=rank, target_types=(nn.Conv2d,))

        in_features = self.model.classifier[1].in_features
        self.model.classifier[1] = nn.Linear(in_features, num_classes)
        for p in self.model.classifier.parameters():
            p.requires_grad = True

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


class ViTB16LoRA(nn.Module):
    def __init__(self, num_classes: int = 10, rank: int = 8, pretrained: bool = True):
        super().__init__()
        weights = ViT_B_16_Weights.DEFAULT if pretrained else None
        self.model = tvm.vit_b_16(weights=weights)

        freeze_module(self.model)

        # ViT: adapt all Linear layers except classifier head.
        inject_dual_lora(self.model, rank=rank, target_types=(nn.Linear,), exclude_names=("heads",))

        in_features = self.model.heads.head.in_features
        self.model.heads.head = nn.Linear(in_features, num_classes)
        for p in self.model.heads.parameters():
            p.requires_grad = True

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


def build_model(arch: str, num_classes: int = 10, lora_rank: int = 8, pretrained: bool = True) -> nn.Module:
    arch = arch.lower()
    if arch == "cnn":
        return SimpleCNN(num_classes=num_classes)
    if arch == "mobilenetv2":
        return MobileNetV2LoRA(num_classes=num_classes, rank=lora_rank, pretrained=pretrained)
    if arch in {"vit", "vit_b_16", "vitb16"}:
        return ViTB16LoRA(num_classes=num_classes, rank=lora_rank, pretrained=pretrained)
    raise ValueError(f"Unknown arch: {arch}")
# %%
# -----------------------------
# Train / eval utilities
# -----------------------------

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device, max_batches: Optional[int] = None) -> Dict[str, float]:
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    criterion = nn.CrossEntropyLoss()
    for b, (x, y) in enumerate(loader):
        if max_batches is not None and b >= max_batches:
            break
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * y.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total += y.size(0)
    return {
        "loss": total_loss / max(total, 1),
        "acc": total_correct / max(total, 1),
        "n": total,
    }


def get_trainable_keys(model: nn.Module) -> List[str]:
    return [name for name, p in model.named_parameters() if p.requires_grad]


def get_trainable_state(model: nn.Module) -> OrderedDict:
    sd = model.state_dict()
    trainable = set(get_trainable_keys(model))
    return OrderedDict((k, v.detach().clone()) for k, v in sd.items() if k in trainable)


def set_trainable_state(model: nn.Module, new_state: Dict[str, torch.Tensor]) -> None:
    sd = model.state_dict()
    trainable = set(get_trainable_keys(model))
    for k, v in new_state.items():
        if k in sd and k in trainable:
            sd[k] = v.detach().clone().to(sd[k].device)
    model.load_state_dict(sd, strict=False)


def get_full_state(model: nn.Module) -> OrderedDict:
    return OrderedDict((k, v.detach().clone()) for k, v in model.state_dict().items())


def set_full_state(model: nn.Module, state: Dict[str, torch.Tensor]) -> None:
    model.load_state_dict(state, strict=True)


def prox_penalty(model: nn.Module, reference_state: Dict[str, torch.Tensor]) -> torch.Tensor:
    penalty = torch.zeros((), device=next(model.parameters()).device)
    ref = reference_state
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        penalty = penalty + 0.5 * torch.sum((param - ref[name].to(param.device)) ** 2)
    return penalty


def train_one_client(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    epochs: int,
    lr: float,
    criterion: Optional[nn.Module] = None,
    optimizer_name: str = "adam",
    weight_decay: float = 0.0,
    prox_mu: Optional[float] = None,
    reference_state: Optional[Dict[str, torch.Tensor]] = None,
    global_grad: Optional[Dict[str, torch.Tensor]] = None,
) -> Dict[str, float]:
    model.train()
    criterion = criterion or nn.CrossEntropyLoss()
    params = [p for p in model.parameters() if p.requires_grad]
    if optimizer_name.lower() == "sgd":
        optimizer = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    else:
        optimizer = torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)

    initial_loss = 0.0
    total_samples = 0

    for _ in range(epochs):
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)

            if prox_mu is not None and reference_state is not None:
                loss = loss + prox_mu * prox_penalty(model, reference_state)

            # Approximate DANE correction: subtract linearized global-gradient term.
            if global_grad is not None:
                corr = torch.zeros((), device=device)
                for name, param in model.named_parameters():
                    if not param.requires_grad:
                        continue
                    if name in global_grad:
                        corr = corr + torch.sum(global_grad[name].to(device) * param)
                loss = loss - corr

            loss.backward()
            optimizer.step()
            initial_loss += loss.item() * y.size(0)
            total_samples += y.size(0)

    return {"loss": initial_loss / max(total_samples, 1)}


@torch.no_grad()
def average_client_models(client_models: List[nn.Module], client_sizes: List[int], only_trainable: bool = True) -> OrderedDict:
    states = []
    weights = []
    for model, n in zip(client_models, client_sizes):
        sd = get_trainable_state(model) if only_trainable else get_full_state(model)
        states.append(sd)
        weights.append(float(n))
    return average_state_dicts(states, weights=weights)

# %%
# -----------------------------
# KMeans (numpy-only)
# -----------------------------

class SimpleKMeans:
    def __init__(self, n_clusters: int, max_iter: int = 100, tol: float = 1e-4, seed: int = 42):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.seed = seed
        self.cluster_centers_: Optional[np.ndarray] = None

    def fit_predict(self, x: np.ndarray) -> np.ndarray:
        rng = np.random.default_rng(self.seed)
        n = x.shape[0]
        if n < self.n_clusters:
            raise ValueError("n_samples must be >= n_clusters")
        centers = x[rng.choice(n, size=self.n_clusters, replace=False)]
        labels = np.zeros(n, dtype=np.int64)
        for _ in range(self.max_iter):
            dists = ((x[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
            new_labels = dists.argmin(axis=1)
            new_centers = np.zeros_like(centers)
            for k in range(self.n_clusters):
                members = x[new_labels == k]
                if len(members) == 0:
                    new_centers[k] = x[rng.integers(0, n)]
                else:
                    new_centers[k] = members.mean(axis=0)
            shift = np.linalg.norm(new_centers - centers)
            centers = new_centers
            labels = new_labels
            if shift < self.tol:
                break
        self.cluster_centers_ = centers
        return labels

# %%
# -----------------------------
# Strategy base class
# -----------------------------

@dataclass
class RoundMetrics:
    round: int
    train_loss: float
    test_loss: float
    test_acc: float


@dataclass
class ExperimentResult:
    name: str
    history: List[RoundMetrics] = field(default_factory=list)
    final_client_acc: Dict[int, float] = field(default_factory=dict)
    final_cluster_assignments: Optional[Dict[int, int]] = None


class FederatedStrategy:
    def __init__(self, name: str, model: nn.Module, client_data: List[ClientData], testloader: DataLoader, device: torch.device,
                 lr: float, rounds: int, local_epochs: int, max_eval_batches: Optional[int] = None):
        self.name = name
        self.global_model = model
        self.client_data = client_data
        self.testloader = testloader
        self.device = device
        self.lr = lr
        self.rounds = rounds
        self.local_epochs = local_epochs
        self.max_eval_batches = max_eval_batches
        self.num_clients = len(client_data)
        self.client_sizes = [cd.n_train for cd in client_data]
        self.round_history: List[RoundMetrics] = []

    def init_global(self) -> None:
        self.global_model.to(self.device)

    def sample_clients(self, round_idx: int) -> List[int]:
        ids = list(range(self.num_clients))
        if CFG.sample_clients_per_round >= 1.0:
            return ids
        return topk_clients_by_size({i: self.client_sizes[i] for i in ids}, CFG.sample_clients_per_round)

    def evaluate_global(self, round_idx: int) -> Dict[str, float]:
        return evaluate(self.global_model, self.testloader, self.device, max_batches=self.max_eval_batches)

    def client_train(self, client_id: int, reference_state: Optional[Dict[str, torch.Tensor]] = None,
                     prox_mu: Optional[float] = None, global_grad: Optional[Dict[str, torch.Tensor]] = None) -> Tuple[nn.Module, float]:
        local_model = clone_model(self.global_model).to(self.device)
        cd = self.client_data[client_id]
        out = train_one_client(
            local_model,
            cd.trainloader,
            self.device,
            epochs=self.local_epochs,
            lr=self.lr,
            optimizer_name="adam",
            prox_mu=prox_mu,
            reference_state=reference_state,
            global_grad=global_grad,
        )
        return local_model, out["loss"]

    def aggregate(self, local_models: List[nn.Module], client_ids: List[int]) -> None:
        weights = [self.client_sizes[cid] for cid in client_ids]
        new_state = average_client_models(local_models, weights, only_trainable=True)
        set_trainable_state(self.global_model, new_state)

    def run(self) -> ExperimentResult:
        self.init_global()
        for r in range(self.rounds):
            selected = self.sample_clients(r)
            local_models = []
            losses = []
            for cid in selected:
                local_model, loss = self.client_train(cid)
                local_models.append(local_model)
                losses.append(loss)
            self.aggregate(local_models, selected)
            test_metrics = self.evaluate_global(r)
            self.round_history.append(RoundMetrics(
                round=r,
                train_loss=float(np.mean(losses)) if losses else float("nan"),
                test_loss=test_metrics["loss"],
                test_acc=test_metrics["acc"],
            ))
            print(f"[{self.name}] round={r:03d} train_loss={self.round_history[-1].train_loss:.4f} test_acc={test_metrics['acc']:.4f}")
        return ExperimentResult(name=self.name, history=self.round_history)

# %%
# -----------------------------
# FedAvg / FedProx / FedCluster / FedDANE / FeSEM
# -----------------------------

class FedAvgStrategy(FederatedStrategy):
    pass


class FedProxStrategy(FederatedStrategy):
    def __init__(self, *args, proximal_mu: float = 0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.proximal_mu = proximal_mu

    def client_train(self, client_id: int, reference_state: Optional[Dict[str, torch.Tensor]] = None,
                     prox_mu: Optional[float] = None, global_grad: Optional[Dict[str, torch.Tensor]] = None) -> Tuple[nn.Module, float]:
        local_model = clone_model(self.global_model).to(self.device)
        ref = get_trainable_state(self.global_model)
        cd = self.client_data[client_id]
        out = train_one_client(
            local_model,
            cd.trainloader,
            self.device,
            epochs=self.local_epochs,
            lr=self.lr,
            optimizer_name="adam",
            prox_mu=self.proximal_mu,
            reference_state=ref,
            global_grad=None,
        )
        return local_model, out["loss"]


class FedClusterStrategy(FederatedStrategy):
    """Cluster clients once after warmup and then aggregate within clusters."""

    def __init__(self, *args, num_clusters: int = 3, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_clusters = num_clusters
        self.cluster_assignments: Dict[int, int] = {}
        self.cluster_models: Dict[int, OrderedDict] = {}

    def warmup_and_cluster(self) -> None:
        local_models = []
        local_vectors = []
        client_ids = list(range(self.num_clients))
        for cid in client_ids:
            lm, _ = self.client_train(cid)
            local_models.append(lm)
            local_vectors.append(model_vector(lm, only_trainable=True))
        X = np.stack(local_vectors, axis=0)
        kmeans = SimpleKMeans(n_clusters=self.num_clusters, seed=CFG.seed)
        labels = kmeans.fit_predict(X)
        self.cluster_assignments = {cid: int(labels[i]) for i, cid in enumerate(client_ids)}
        for k in range(self.num_clusters):
            members = [cid for cid in client_ids if self.cluster_assignments[cid] == k]
            if not members:
                continue
            member_models = [local_models[cid] for cid in members]
            weights = [self.client_sizes[cid] for cid in members]
            self.cluster_models[k] = average_client_models(member_models, weights, only_trainable=True)
        print(f"[FedCluster] assignments={self.cluster_assignments}")

    def run(self) -> ExperimentResult:
        self.init_global()
        self.warmup_and_cluster()

        # Use a single global model initialized from averaged cluster models.
        if self.cluster_models:
            averaged = average_state_dicts(list(self.cluster_models.values()))
            set_trainable_state(self.global_model, averaged)

        for r in range(self.rounds):
            local_models = []
            client_ids = []
            for cid in range(self.num_clients):
                cluster_id = self.cluster_assignments[cid]
                if cluster_id in self.cluster_models:
                    set_trainable_state(self.global_model, self.cluster_models[cluster_id])
                lm, loss = self.client_train(cid)
                local_models.append(lm)
                client_ids.append(cid)
            # Aggregate per cluster and update the global model with the mean cluster state.
            cluster_states = []
            for k in range(self.num_clusters):
                members = [i for i in client_ids if self.cluster_assignments[i] == k]
                if not members:
                    continue
                member_models = [local_models[i] for i, cid in enumerate(client_ids) if self.cluster_assignments[cid] == k]
                weights = [self.client_sizes[cid] for cid in members]
                cluster_state = average_client_models(member_models, weights, only_trainable=True)
                self.cluster_models[k] = cluster_state
                cluster_states.append(cluster_state)
            if cluster_states:
                set_trainable_state(self.global_model, average_state_dicts(cluster_states))
            test_metrics = self.evaluate_global(r)
            self.round_history.append(RoundMetrics(r, float("nan"), test_metrics["loss"], test_metrics["acc"]))
            print(f"[FedCluster] round={r:03d} test_acc={test_metrics['acc']:.4f}")
        return ExperimentResult(name=self.name, history=self.round_history, final_cluster_assignments=self.cluster_assignments)


class FedDaneStrategy(FederatedStrategy):
    """Approximate FedDANE: server keeps a gradient estimate built from the last round."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.global_grad: Optional[Dict[str, torch.Tensor]] = None

    def estimate_global_grad(self, client_models: List[nn.Module], client_ids: List[int]) -> Dict[str, torch.Tensor]:
        # Gradient surrogate: mean signed deviation from the current model.
        base_state = get_trainable_state(self.global_model)
        grads: Dict[str, torch.Tensor] = {k: torch.zeros_like(v) for k, v in base_state.items()}
        for lm in client_models:
            st = get_trainable_state(lm)
            for k in grads:
                grads[k] += (base_state[k] - st[k])
        for k in grads:
            grads[k] /= max(1, len(client_models))
        return grads

    def run(self) -> ExperimentResult:
        self.init_global()
        for r in range(self.rounds):
            selected = self.sample_clients(r)
            local_models = []
            losses = []
            for cid in selected:
                lm, loss = self.client_train(cid, global_grad=self.global_grad)
                local_models.append(lm)
                losses.append(loss)
            self.global_grad = self.estimate_global_grad(local_models, selected)
            self.aggregate(local_models, selected)
            test_metrics = self.evaluate_global(r)
            self.round_history.append(RoundMetrics(r, float(np.mean(losses)) if losses else float("nan"), test_metrics["loss"], test_metrics["acc"]))
            print(f"[FedDANE] round={r:03d} test_acc={test_metrics['acc']:.4f}")
        return ExperimentResult(name=self.name, history=self.round_history)


class FeSEMStrategy(FederatedStrategy):
    """Multi-center EM-style strategy with soft assignments."""

    def __init__(self, *args, num_clusters: int = 3, temperature: float = 1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_clusters = num_clusters
        self.temperature = temperature
        self.cluster_models: List[OrderedDict] = []
        self.assignments: Dict[int, int] = {}

    def initialize_clusters(self) -> None:
        # Start from random client models.
        seed_clients = list(range(min(self.num_clusters, self.num_clients)))
        self.cluster_models = []
        for cid in seed_clients:
            lm, _ = self.client_train(cid)
            self.cluster_models.append(get_trainable_state(lm))
        while len(self.cluster_models) < self.num_clusters:
            self.cluster_models.append(get_trainable_state(self.global_model))

    def soft_assign(self, client_state: OrderedDict) -> Tuple[int, np.ndarray]:
        sims = []
        client_vec = np.concatenate([v.detach().cpu().flatten().numpy() for v in client_state.values()])
        for cm in self.cluster_models:
            cluster_vec = np.concatenate([v.detach().cpu().flatten().numpy() for v in cm.values()])
            dist = np.linalg.norm(client_vec - cluster_vec)
            sims.append(-dist)
        sims = np.array(sims, dtype=np.float32)
        probs = np.exp((sims - sims.max()) / max(self.temperature, 1e-8))
        probs = probs / probs.sum()
        return int(probs.argmax()), probs

    def run(self) -> ExperimentResult:
        self.init_global()
        self.initialize_clusters()
        for r in range(self.rounds):
            client_states: List[OrderedDict] = []
            client_ids: List[int] = []
            probs_all: List[np.ndarray] = []
            for cid in range(self.num_clients):
                # Use the closest cluster model as the starting point.
                if self.cluster_models:
                    chosen = cid % len(self.cluster_models)
                    set_trainable_state(self.global_model, self.cluster_models[chosen])
                lm, _ = self.client_train(cid)
                st = get_trainable_state(lm)
                assigned, probs = self.soft_assign(st)
                self.assignments[cid] = assigned
                client_states.append(st)
                client_ids.append(cid)
                probs_all.append(probs)

            new_clusters: List[OrderedDict] = []
            for k in range(self.num_clusters):
                members = [i for i, cid in enumerate(client_ids) if self.assignments[cid] == k]
                if not members:
                    new_clusters.append(self.cluster_models[k])
                    continue
                member_states = [client_states[i] for i in members]
                weights = [self.client_sizes[client_ids[i]] for i in members]
                new_clusters.append(average_state_dicts(member_states, weights=weights))
            self.cluster_models = new_clusters
            set_trainable_state(self.global_model, average_state_dicts(self.cluster_models))
            test_metrics = self.evaluate_global(r)
            self.round_history.append(RoundMetrics(r, float("nan"), test_metrics["loss"], test_metrics["acc"]))
            print(f"[FeSEM] round={r:03d} test_acc={test_metrics['acc']:.4f}")
        return ExperimentResult(name=self.name, history=self.round_history, final_cluster_assignments=self.assignments)

# %%
# -----------------------------
# cFedLoRA
# -----------------------------

class CFedLoRAStrategy(FederatedStrategy):
    """Cluster clients based on LoRA deltas after round 0, then train within fixed clusters.

    This implements the core idea of cFedLoRA in a practical, notebook-friendly way:
    - warmup one local round
    - cluster client LoRA updates using K-means
    - perform cluster-wise aggregation
    - optional personalization
    """

    def __init__(self, *args, num_clusters: int = 3, lora_name_filter: Optional[Callable[[str], bool]] = None,
                 personalization: bool = True, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_clusters = num_clusters
        self.cluster_assignments: Dict[int, int] = {}
        self.cluster_models: Dict[int, OrderedDict] = {}
        self.personalization = personalization
        self.lora_name_filter = lora_name_filter or (lambda name: "lora_" in name)

    def lora_state_vector(self, model: nn.Module) -> np.ndarray:
        vecs = []
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            if self.lora_name_filter(name):
                vecs.append(param.detach().cpu().flatten().numpy())
        if not vecs:
            return np.zeros(1, dtype=np.float32)
        return np.concatenate(vecs).astype(np.float32)

    def warmup_and_cluster(self) -> Dict[int, int]:
        client_models = []
        vectors = []
        client_ids = list(range(self.num_clients))
        for cid in client_ids:
            lm, _ = self.client_train(cid)
            client_models.append(lm)
            vectors.append(self.lora_state_vector(lm))
        X = np.stack(vectors, axis=0)
        km = SimpleKMeans(n_clusters=self.num_clusters, seed=CFG.seed)
        labels = km.fit_predict(X)
        assignments = {cid: int(labels[i]) for i, cid in enumerate(client_ids)}
        self.cluster_assignments = assignments

        for k in range(self.num_clusters):
            members = [cid for cid in client_ids if assignments[cid] == k]
            if not members:
                continue
            member_models = [client_models[cid] for cid in members]
            weights = [self.client_sizes[cid] for cid in members]
            self.cluster_models[k] = average_client_models(member_models, weights, only_trainable=True)
        return assignments

    def run(self) -> ExperimentResult:
        self.init_global()
        self.warmup_and_cluster()
        if self.cluster_models:
            set_trainable_state(self.global_model, average_state_dicts(list(self.cluster_models.values())))

        for r in range(self.rounds):
            cluster_local_states: Dict[int, List[OrderedDict]] = {k: [] for k in range(self.num_clusters)}
            cluster_sizes: Dict[int, List[int]] = {k: [] for k in range(self.num_clusters)}
            for cid in range(self.num_clients):
                k = self.cluster_assignments[cid]
                if k in self.cluster_models:
                    set_trainable_state(self.global_model, self.cluster_models[k])
                lm, _ = self.client_train(cid)
                cluster_local_states[k].append(get_trainable_state(lm))
                cluster_sizes[k].append(self.client_sizes[cid])

            for k in range(self.num_clusters):
                if cluster_local_states[k]:
                    self.cluster_models[k] = average_state_dicts(cluster_local_states[k], cluster_sizes[k])

            if self.cluster_models:
                set_trainable_state(self.global_model, average_state_dicts(list(self.cluster_models.values())))

            test_metrics = self.evaluate_global(r)
            self.round_history.append(RoundMetrics(r, float("nan"), test_metrics["loss"], test_metrics["acc"]))
            print(f"[cFedLoRA] round={r:03d} test_acc={test_metrics['acc']:.4f}")

        if self.personalization:
            self.personalize_all_clients()

        return ExperimentResult(name=self.name, history=self.round_history, final_cluster_assignments=self.cluster_assignments)

    def personalize_all_clients(self) -> None:
        print("[cFedLoRA] personalization stage")
        base_cluster_model = average_state_dicts(list(self.cluster_models.values())) if self.cluster_models else get_trainable_state(self.global_model)
        for cid in range(self.num_clients):
            k = self.cluster_assignments[cid]
            if k in self.cluster_models:
                set_trainable_state(self.global_model, self.cluster_models[k])
            cd = self.client_data[cid]
            local_model = clone_model(self.global_model).to(self.device)
            train_one_client(
                local_model,
                cd.personalloader,
                self.device,
                epochs=CFG.finetune_epochs,
                lr=self.lr,
                optimizer_name="adam",
            )
            # Store personalized model back into cluster model as a simple practical variant.
            self.cluster_models[k] = get_trainable_state(local_model)

# %%
# -----------------------------
# Baseline factory
# -----------------------------

def make_strategy(strategy_name: str, arch: str, client_data: List[ClientData], testloader: DataLoader, device: torch.device,
                  lora_rank: int = 8, num_clusters: int = 3, pretrained: bool = True) -> FederatedStrategy:
    strategy_name = strategy_name.lower()
    arch = arch.lower()

    model = build_model(arch, num_classes=CFG.num_classes, lora_rank=lora_rank, pretrained=pretrained)
    model.to(device)

    if arch == "cnn":
        lr = CFG.lr_cnn
    elif arch == "mobilenetv2":
        lr = CFG.lr_mobilenet
    else:
        lr = CFG.lr_vit

    common_kwargs = dict(
        name=f"{strategy_name}-{arch}",
        model=model,
        client_data=client_data,
        testloader=testloader,
        device=device,
        lr=lr,
        rounds=CFG.rounds,
        local_epochs=CFG.local_epochs,
        max_eval_batches=CFG.max_eval_batches,
    )

    if strategy_name == "fedavg":
        return FedAvgStrategy(**common_kwargs)
    if strategy_name == "fedprox":
        return FedProxStrategy(**common_kwargs, proximal_mu=CFG.proximal_mu)
    if strategy_name == "fedcluster":
        return FedClusterStrategy(**common_kwargs, num_clusters=num_clusters)
    if strategy_name == "feddane":
        return FedDaneStrategy(**common_kwargs)
    if strategy_name == "fesem":
        return FeSEMStrategy(**common_kwargs, num_clusters=num_clusters)
    if strategy_name == "cfedlora":
        return CFedLoRAStrategy(**common_kwargs, num_clusters=num_clusters, personalization=True)
    if strategy_name == "fedavg_lora":
        return FedAvgStrategy(**common_kwargs)

    raise ValueError(f"Unknown strategy: {strategy_name}")

# %%
# -----------------------------
# Experiment runner
# -----------------------------

@dataclass
class RunSpec:
    strategy: str
    arch: str
    lora_rank: int = 8
    num_clusters: int = 3
    pretrained: bool = True


def run_experiment(spec: RunSpec, client_data: List[ClientData], testloader: DataLoader) -> ExperimentResult:
    strategy = make_strategy(
        strategy_name=spec.strategy,
        arch=spec.arch,
        client_data=client_data,
        testloader=testloader,
        device=DEVICE,
        lora_rank=spec.lora_rank,
        num_clusters=spec.num_clusters,
        pretrained=spec.pretrained,
    )
    result = strategy.run()
    return result


def plot_histories(results: List[ExperimentResult], title: str = "Test Accuracy") -> None:
    plt.figure(figsize=(10, 6))
    for res in results:
        xs = [r.round for r in res.history]
        ys = [r.test_acc for r in res.history]
        plt.plot(xs, ys, label=res.name)
    plt.title(title)
    plt.xlabel("Round")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# %%
# -----------------------------
# Data loading example
# -----------------------------

# Choose the model family for the experiment:
# - "cnn"
# - "mobilenetv2"
# - "vit_b_16"
MODEL_NAME = "mobilenetv2"

client_data, testloader, train_labels = build_federated_cifar10(
    model_name=MODEL_NAME,
    num_clients=CFG.num_clients,
    alpha=CFG.alpha,
    batch_size=CFG.batch_size,
    test_fraction_per_client=CFG.test_fraction_per_client,
    personal_fraction_per_client=CFG.personal_fraction_per_client,
    seed=CFG.seed,
    download=CFG.download_data,
)

print("Client sample sizes:", [cd.n_train for cd in client_data])
print("Client label histograms:")
for i, cd in enumerate(client_data):
    print(i, cd.label_hist)

# %%
# -----------------------------
# Sanity checks for model construction
# -----------------------------

cnn = build_model("cnn", num_classes=CFG.num_classes, pretrained=False)
mnv2 = build_model("mobilenetv2", num_classes=CFG.num_classes, lora_rank=CFG.lora_rank, pretrained=CFG.pretrained)
vit = build_model("vit_b_16", num_classes=CFG.num_classes, lora_rank=CFG.lora_rank, pretrained=CFG.pretrained)

print("CNN params total/trainable:", count_parameters(cnn, False), count_parameters(cnn, True))
print("MobileNetV2 params total/trainable:", count_parameters(mnv2, False), count_parameters(mnv2, True))
print("ViT-B/16 params total/trainable:", count_parameters(vit, False), count_parameters(vit, True))

# %%
# -----------------------------
# Example experiment plan
# -----------------------------

# The default configuration below is intentionally conservative for a notebook.
# You can increase rounds/local_epochs once the pipeline is verified.
#
# Suggested set of experiments:
#
# CNN backbone:
#   - FedAvg
#   - FedProx
#   - FedCluster
#   - FedDANE
#   - FeSEM
#
# MobileNetV2 backbone:
#   - FedAvg
#   - FedProx
#   - FedCluster
#   - FedDANE
#   - FeSEM
#   - FedAvg+LoRA
#   - cFedLoRA
#
# ViT-B/16 backbone:
#   - FedAvg
#   - FedProx
#   - FedCluster
#   - FedDANE
#   - FeSEM
#   - FedAvg+LoRA
#   - cFedLoRA
#
# To keep the notebook interactive, run a small subset first:
#
# results = []
# for spec in [
#     RunSpec("fedavg", "cnn", pretrained=False),
#     RunSpec("fedprox", "cnn", pretrained=False),
#     RunSpec("fedcluster", "cnn", pretrained=False),
# ]:
#     results.append(run_experiment(spec, client_data, testloader))
# plot_histories(results, title="CNN baselines on CIFAR-10")

# %%
# -----------------------------
# Helper to export round histories
# -----------------------------

def history_to_list(result: ExperimentResult) -> List[Dict[str, float]]:
    return [
        {
            "round": r.round,
            "train_loss": r.train_loss,
            "test_loss": r.test_loss,
            "test_acc": r.test_acc,
        }
        for r in result.history
    ]


# %%
# -----------------------------
# Flower integration layer (CIFAR-10, partition-id driven)
# -----------------------------

# Flower uses a client/server split. The client loads its own partition by partition-id,
# which is provided by the Flower runtime via Context.node_config.
# This keeps CIFAR-10 as the dataset while making the orchestration Flower-native.

FLOWER_NUM_PARTITIONS = CFG.num_clients
FLOWER_MODEL_NAME = MODEL_NAME


def load_flower_partition(partition_id: int,
                          num_partitions: int = FLOWER_NUM_PARTITIONS,
                          model_name: str = FLOWER_MODEL_NAME,
                          batch_size: int = CFG.batch_size,
                          seed: int = CFG.seed,
                          download: bool = CFG.download_data) -> ClientData:
    """Deterministically reconstruct the local partition for a given Flower client.

    Each simulated/deployed Flower node can call this function independently.
    The returned loaders follow the same CIFAR-10 + Dirichlet(alpha) split used elsewhere
    in the notebook.
    """
    trainset_raw, _ = load_cifar10(download=download)
    train_tf, test_tf = get_transforms(model_name)
    labels = np.array(trainset_raw.targets)
    client_map = dirichlet_partition_indices(labels, num_clients=num_partitions, alpha=CFG.alpha, seed=seed)

    # Defensive modulo so the function is robust even if the runtime sends a larger id.
    cid = int(partition_id) % num_partitions
    train_idx, personal_idx = split_client_indices(client_map[cid], personal_fraction=CFG.personal_fraction_per_client, seed=seed + cid)
    n_val = max(1, int(round(len(train_idx) * CFG.test_fraction_per_client))) if len(train_idx) > 1 else 0
    rng = np.random.default_rng(seed + 10_000 + cid)
    rng.shuffle(train_idx)
    val_idx = train_idx[:n_val]
    main_train_idx = train_idx[n_val:]

    train_ds = IndexedDataset(trainset_raw, main_train_idx, transform=train_tf)
    val_ds = IndexedDataset(trainset_raw, val_idx, transform=test_tf)
    personal_ds = IndexedDataset(trainset_raw, personal_idx, transform=test_tf)

    hist = np.bincount(labels[np.array(client_map[cid])], minlength=CFG.num_classes)
    return ClientData(
        trainloader=DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0),
        valloader=DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0),
        personalloader=DataLoader(personal_ds, batch_size=batch_size, shuffle=True, num_workers=0),
        n_train=len(train_ds),
        n_val=len(val_ds),
        n_personal=len(personal_ds),
        label_hist=hist,
    )

class FlowerCIFARClient(NumPyClient):
    """
    Flower client for CIFAR-10.

    This version serializes only trainable parameters by default.
    That is the right next step for LoRA / dual-LoRA experiments, because:
      - frozen backbone weights do not need to be retransmitted every round
      - server-side aggregation can later be specialized on the trainable subset
      - the parameter order stays consistent across all clients
    """

    def __init__(
        self,
        partition_id: int,
        model_name: str,
        strategy_name: str,
        lora_rank: int = CFG.lora_rank,
        pretrained: bool = CFG.pretrained,
    ):
        self.partition_id = partition_id
        self.model_name = model_name
        self.strategy_name = strategy_name.lower()
        self.lora_rank = lora_rank
        self.pretrained = pretrained

        self.model = build_model(
            model_name,
            num_classes=CFG.num_classes,
            lora_rank=lora_rank,
            pretrained=pretrained,
        ).to(DEVICE)

        self.data = load_flower_partition(
            partition_id,
            num_partitions=FLOWER_NUM_PARTITIONS,
            model_name=model_name,
        )

        # Fixed parameter order for all trainable tensors.
        self.trainable_names = [
            name for name, p in self.model.named_parameters() if p.requires_grad
        ]

    def _get_trainable_ndarrays(self) -> List[np.ndarray]:
        state = self.model.state_dict()
        return [state[name].detach().cpu().numpy() for name in self.trainable_names]

    def _set_trainable_ndarrays(self, ndarrays: List[np.ndarray]) -> None:
        if len(ndarrays) != len(self.trainable_names):
            raise ValueError(
                f"Expected {len(self.trainable_names)} tensors, got {len(ndarrays)}"
            )

        state = self.model.state_dict()
        for name, array in zip(self.trainable_names, ndarrays):
            tensor = torch.tensor(array, dtype=state[name].dtype)
            state[name] = tensor
        self.model.load_state_dict(state, strict=False)

    def get_parameters(self, config=None):
        return self._get_trainable_ndarrays()

    def set_parameters(self, parameters: List[np.ndarray]) -> None:
        self._set_trainable_ndarrays(parameters)

    def fit(self, parameters, config):
        self.set_parameters(parameters)

        lr = float(
            config.get(
                "lr",
                CFG.lr_cnn
                if self.model_name == "cnn"
                else CFG.lr_mobilenet
                if self.model_name == "mobilenetv2"
                else CFG.lr_vit,
            )
        )
        epochs = int(config.get("local_epochs", CFG.local_epochs))

        prox_mu = None
        reference_state = None
        if self.strategy_name == "fedprox":
            prox_mu = float(config.get("prox_mu", CFG.proximal_mu))
            reference_state = {
                name: tensor.detach().clone()
                for name, tensor in self.model.state_dict().items()
                if name in self.trainable_names
            }

        train_one_client(
            self.model,
            self.data.trainloader,
            DEVICE,
            epochs=epochs,
            lr=lr,
            optimizer_name="adam",
            prox_mu=prox_mu,
            reference_state=reference_state,
        )

        # Return only trainable tensors, in the same order as get_parameters.
        return self.get_parameters(), self.data.n_train, {}

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        metrics = evaluate(self.model, self.data.valloader, DEVICE, max_batches=None)
        return float(metrics["loss"]), self.data.n_val, {"acc": float(metrics["acc"])}
# Flower client factory expected by ClientApp

def client_fn(context: Context):
    partition_id = int(context.node_config.get("partition-id", 0))
    model_name = str(context.run_config.get("model-name", FLOWER_MODEL_NAME))
    strategy_name = str(context.run_config.get("strategy-name", "fedavg"))
    lora_rank = int(context.run_config.get("lora-rank", CFG.lora_rank))
    pretrained = bool(context.run_config.get("pretrained", CFG.pretrained))
    return FlowerCIFARClient(
        partition_id=partition_id,
        model_name=model_name,
        strategy_name=strategy_name,
        lora_rank=lora_rank,
        pretrained=pretrained,
    ).to_client()


client_app = ClientApp(client_fn=client_fn)


# Server side: Flower-native orchestration for the baseline FedAvg path.
# The heavier algorithm-specific baselines remain available in the pure-PyTorch
# strategy classes above and can be wired into Flower in the next pass.

def server_fn(context: Context):
    from flwr.server import ServerAppComponents, ServerConfig
    from flwr.server.strategy import FedAvg as FlowerFedAvg

    num_rounds = int(context.run_config.get("num-rounds", CFG.rounds))
    lr = float(context.run_config.get("lr", CFG.lr_cnn if FLOWER_MODEL_NAME == "cnn" else CFG.lr_mobilenet if FLOWER_MODEL_NAME == "mobilenetv2" else CFG.lr_vit))
    strategy = FlowerFedAvg(
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=CFG.num_clients,
        min_evaluate_clients=CFG.num_clients,
        min_available_clients=CFG.num_clients,
    )
    server_config = ServerConfig(num_rounds=num_rounds)
    return ServerAppComponents(strategy=strategy, server_config=server_config)


server_app = ServerApp(server_fn=server_fn)


def run_flower_simulation(num_supernodes: int = CFG.num_clients, verbose_logging: bool = True) -> None:
    """Run the Flower simulation runtime for the CIFAR-10 setup."""
    run_simulation(
        server_app=server_app,
        client_app=client_app,
        num_supernodes=num_supernodes,
        verbose_logging=verbose_logging,
    )


print("Notebook scaffold loaded successfully.")


In [ ]:
# %%
# -----------------------------
# Full experiment matrix runner
# -----------------------------

def build_client_data_for_arch(arch: str) -> Tuple[List[ClientData], DataLoader]:
    arch = arch.lower()
    client_data, testloader, _ = build_federated_cifar10(
        model_name=arch,
        num_clients=CFG.num_clients,
        alpha=CFG.alpha,
        batch_size=CFG.batch_size,
        test_fraction_per_client=CFG.test_fraction_per_client,
        personal_fraction_per_client=CFG.personal_fraction_per_client,
        seed=CFG.seed,
        download=CFG.download_data,
    )
    return client_data, testloader


def run_suite_for_arch(
    arch: str,
    strategies: List[str],
    lora_rank: int = 8,
    num_clusters: int = 3,
    pretrained: bool = True,
) -> List[ExperimentResult]:
    client_data, testloader = build_client_data_for_arch(arch)
    results: List[ExperimentResult] = []

    for strategy_name in strategies:
        print("\n" + "=" * 80)
        print(f"Running strategy={strategy_name} | arch={arch} | rank={lora_rank} | K={num_clusters}")
        print("=" * 80)

        spec = RunSpec(
            strategy=strategy_name,
            arch=arch,
            lora_rank=lora_rank,
            num_clusters=num_clusters,
            pretrained=pretrained,
        )
        result = run_experiment(spec, client_data, testloader)
        results.append(result)

    return results


def run_all_paper_experiments() -> Dict[str, List[ExperimentResult]]:
    """
    Runs the full baseline set for CIFAR-10.

    Recommended order:
    1) CNN baselines
    2) MobileNetV2 baselines
    3) ViT-B/16 baselines
    """
    all_results: Dict[str, List[ExperimentResult]] = {}

    cnn_strategies = ["fedavg", "fedprox", "fedcluster", "feddane", "fesem"]
    mnv2_strategies = ["fedavg", "fedprox", "fedcluster", "feddane", "fesem", "fedavg_lora", "cfedlora"]
    vit_strategies = ["fedavg", "fedprox", "fedcluster", "feddane", "fesem", "fedavg_lora", "cfedlora"]

    # CNN uses a non-pretrained lightweight backbone in this notebook
    all_results["cnn"] = run_suite_for_arch(
        arch="cnn",
        strategies=cnn_strategies,
        lora_rank=CFG.lora_rank,
        num_clusters=CFG.num_clusters,
        pretrained=False,
    )

    # MobileNetV2 + LoRA
    all_results["mobilenetv2"] = run_suite_for_arch(
        arch="mobilenetv2",
        strategies=mnv2_strategies,
        lora_rank=CFG.lora_rank,
        num_clusters=CFG.num_clusters,
        pretrained=True,
    )

    # ViT-B/16 + LoRA
    all_results["vit_b_16"] = run_suite_for_arch(
        arch="vit_b_16",
        strategies=vit_strategies,
        lora_rank=CFG.lora_rank,
        num_clusters=CFG.num_clusters,
        pretrained=True,
    )

    return all_results


def plot_suite_results(all_results: Dict[str, List[ExperimentResult]]) -> None:
    for arch, results in all_results.items():
        plt.figure(figsize=(10, 6))
        for res in results:
            xs = [r.round for r in res.history]
            ys = [r.test_acc for r in res.history]
            plt.plot(xs, ys, label=res.name)
        plt.title(f"Test Accuracy on CIFAR-10 — {arch}")
        plt.xlabel("Round")
        plt.ylabel("Accuracy")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


def print_final_summary(all_results: Dict[str, List[ExperimentResult]]) -> None:
    for arch, results in all_results.items():
        print(f"\n{arch.upper()}")
        for res in results:
            if not res.history:
                continue
            best_acc = max(r.test_acc for r in res.history)
            final_acc = res.history[-1].test_acc
            print(f"{res.name:20s} | final_acc={final_acc:.4f} | best_acc={best_acc:.4f}")

# %%
# -----------------------------
# Flower integration layer
# -----------------------------

from flwr.common import Context
from flwr.server import ServerApp, ServerAppComponents, ServerConfig
from flwr.server.strategy import FedAvg


def get_on_fit_config_fn(
    strategy_name: str,
    model_name: str,
    lora_rank: int,
) :
    def fit_config(server_round: int) -> Dict[str, str]:
        lr = (
            CFG.lr_cnn
            if model_name == "cnn"
            else CFG.lr_mobilenet
            if model_name == "mobilenetv2"
            else CFG.lr_vit
        )
        return {
            "server_round": str(server_round),
            "strategy-name": strategy_name,
            "model-name": model_name,
            "lora-rank": str(lora_rank),
            "local_epochs": str(CFG.local_epochs),
            "lr": str(lr),
            "prox_mu": str(CFG.proximal_mu),
            "num_clusters": str(CFG.num_clusters),
            "personalization": "1" if strategy_name == "cfedlora" else "0",
        }

    return fit_config


def get_on_evaluate_config_fn(strategy_name: str, model_name: str):
    def evaluate_config(server_round: int) -> Dict[str, str]:
        return {
            "server_round": str(server_round),
            "strategy-name": strategy_name,
            "model-name": model_name,
        }

    return evaluate_config


def make_client_fn(
    strategy_name: str,
    model_name: str,
    lora_rank: int,
    pretrained: bool,
):
    def client_fn(context: Context):
        partition_id = int(context.node_config.get("partition-id", 0))
        return FlowerCIFARClient(
            partition_id=partition_id,
            model_name=model_name,
            strategy_name=strategy_name,
            lora_rank=lora_rank,
            pretrained=pretrained,
        ).to_client()

    return client_fn


def make_server_fn(
    strategy_name: str,
    model_name: str,
    num_rounds: int,
    lora_rank: int,
    pretrained: bool,
):
    def server_fn(context: Context):
        strategy = FedAvg(
            fraction_fit=1.0,
            fraction_evaluate=1.0,
            min_fit_clients=CFG.num_clients,
            min_evaluate_clients=CFG.num_clients,
            min_available_clients=CFG.num_clients,
            on_fit_config_fn=get_on_fit_config_fn(strategy_name, model_name, lora_rank),
            on_evaluate_config_fn=get_on_evaluate_config_fn(strategy_name, model_name),
        )

        server_config = ServerConfig(num_rounds=num_rounds)
        return ServerAppComponents(
            strategy=strategy,
            server_config=server_config,
        )

    return server_fn


def build_flower_apps(
    strategy_name: str,
    model_name: str,
    num_rounds: int = CFG.rounds,
    lora_rank: int = CFG.lora_rank,
    pretrained: bool = CFG.pretrained,
):
    server_app = ServerApp(
        server_fn=make_server_fn(
            strategy_name=strategy_name,
            model_name=model_name,
            num_rounds=num_rounds,
            lora_rank=lora_rank,
            pretrained=pretrained,
        )
    )
    client_app = ClientApp(
        client_fn=make_client_fn(
            strategy_name=strategy_name,
            model_name=model_name,
            lora_rank=lora_rank,
            pretrained=pretrained,
        )
    )
    return server_app, client_app


def run_flower_simulation(
    strategy_name: str,
    model_name: str,
    num_supernodes: int = CFG.num_clients,
    num_rounds: int = CFG.rounds,
    lora_rank: int = CFG.lora_rank,
    pretrained: bool = CFG.pretrained,
    verbose_logging: bool = True,
):
    server_app, client_app = build_flower_apps(
        strategy_name=strategy_name,
        model_name=model_name,
        num_rounds=num_rounds,
        lora_rank=lora_rank,
        pretrained=pretrained,
    )

    run_simulation(
        server_app=server_app,
        client_app=client_app,
        num_supernodes=num_supernodes,
        verbose_logging=verbose_logging,
    )

In [ ]:
CFG.rounds = 3
CFG.local_epochs = 1
CFG.finetune_epochs = 1

quick_results = run_suite_for_arch(
    arch="mobilenetv2",
    strategies=["fedavg", "fedprox", "fedcluster", "feddane", "fesem", "fedavg_lora", "cfedlora"],
    lora_rank=8,
    num_clusters=3,
    pretrained=True,
)

plot_histories(quick_results, title="Quick sanity check — MobileNetV2 on CIFAR-10")
print_final_summary({"mobilenetv2": quick_results})